In [ ]:
!pip install transformers datasets torch accelerate -q

import torch
print("GPU Available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

GPU Available: True
Device: Tesla T4


In [ ]:

import pandas as pd
import requests
import io

# Download dataset directly in Colab
url = "https://huggingface.co/datasets/bdotloh/empathetic-dialogues-contexts/resolve/main/train.csv"
response = requests.get(url)
df = pd.read_csv(io.StringIO(response.text))

print("Dataset loaded!")
print(f"Total rows: {len(df)}")
print(df.columns.tolist())
df.head()

Dataset loaded!
Total rows: 19209
['Unnamed: 0', 'situation', 'emotion']


,Unnamed: 0,situation,emotion
0,0,I remember going to the fireworks with my best...,sentimental
1,1,i used to scare for darkness,afraid
2,2,I showed a guy how to run a good bead in weldi...,proud
3,3,I have always been loyal to my wife.,faithful
4,4,A recent job interview that I had made me feel...,terrified


In [ ]:
from transformers import GPT2Tokenizer

# Load DistilGPT2 tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('distilgpt2')
tokenizer.pad_token = tokenizer.eos_token


def prepare_text(row):
    return f"Emotion: {row['emotion']}\nSituation: {row['situation']}\nResponse: I understand you're feeling {row['emotion']}. {row['situation']} That must have been really difficult. You are not alone in this.<|endoftext|>"

df['text'] = df.apply(prepare_text, axis=1)

train_df = df.sample(500, random_state=42).reset_index(drop=True)

print("Sample prepared text:")
print(train_df['text'][0])
print(f"\nTotal training examples: {len(train_df)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Sample prepared text:
Emotion: embarrassed
Situation: My social anxiety acted up and I failed at doing basic math in my head when paying the cashier and looked like an idiot!
Response: I understand you're feeling embarrassed. My social anxiety acted up and I failed at doing basic math in my head when paying the cashier and looked like an idiot! That must have been really difficult. You are not alone in this.<|endoftext|>

Total training examples: 500


In [ ]:
from torch.utils.data import Dataset
import torch

class EmpatheticDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=128):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors='pt'
        )

    def __len__(self):
        return len(self.encodings['input_ids'])

    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': self.encodings['input_ids'][idx]
        }

# Create dataset
texts = train_df['text'].tolist()
train_dataset = EmpatheticDataset(texts, tokenizer)

print(f"Dataset size: {len(train_dataset)}")
print("Dataset ready!")

Dataset size: 500
Dataset ready!


In [ ]:
from transformers import GPT2LMHeadModel, Trainer, TrainingArguments

# Load DistilGPT2 model
model = GPT2LMHeadModel.from_pretrained('distilgpt2')
model.resize_token_embeddings(len(tokenizer))

# Training arguments
training_args = TrainingArguments(
    output_dir='./mental_health_model',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    save_steps=100,
    logging_steps=50,
    prediction_loss_only=True,
    use_cpu=False
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

# Start training
print("Training started — please wait...")
trainer.train()
print("Training complete!")

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
50,1.345468
100,0.621005
150,0.610683


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete!


In [ ]:

model.save_pretrained('./mental_health_model')
tokenizer.save_pretrained('./mental_health_model')

print("Model saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully!


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

def mental_health_chatbot(user_input, max_length=150):
    prompt = f"Situation: {user_input}\nResponse:"

    inputs = tokenizer.encode(prompt, return_tensors='pt')
    inputs = inputs.to(device)

    attention_mask = torch.ones(inputs.shape, device=device)

    with torch.no_grad():
        outputs = model.generate(
            inputs,
            attention_mask=attention_mask,
            max_length=max_length,
            num_return_sequences=1,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "Response:" in response:
        response = response.split("Response:")[-1].strip()

    return response

print("Chatbot ready!")

Chatbot ready!


In [ ]:
test_inputs = [
    "I feel so stressed and overwhelmed",
    "I'm feeling really anxious today",
    "I feel so lonely and sad",
    "I'm exhausted and burnt out"
]

for user_input in test_inputs:
    print(f"User: {user_input}")
    print(f"Bot: {mental_health_chatbot(user_input)}")
    print("-" * 60)

User: I feel so stressed and overwhelmed
Bot: I understand you're feeling stressed and overwhelmed. That must have been really difficult. You are not alone in this.
------------------------------------------------------------
User: I'm feeling really anxious today
Bot: I understand you're feeling really anxious today. I'm feeling really anxious today That must have been really difficult. You are not alone in this.
------------------------------------------------------------
User: I feel so lonely and sad
Bot: I understand you're feeling lonely and sad. I understand you're feeling lonely and sad That must have been really difficult. You are not alone in this.
------------------------------------------------------------
User: I'm exhausted and burnt out
Bot: I understand you're feeling exhausted and burnt out. That must have been really difficult. You are not alone in this.
------------------------------------------------------------


In [ ]:
import requests

API_KEY = "your api key"

HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

mental_health_prompt = """You are a warm, friendly and
supportive mental health assistant. When someone shares
their situation:
1. First acknowledge their feelings with empathy
2. Then give 3-4 practical solutions they can try
3. Suggest when to seek professional help
4. Keep tone friendly, warm and encouraging
5. Use simple language
6. End with a motivating line

Never diagnose. Always be gentle and supportive."""

def mental_health_chatbot_v2(user_input):
    payload = {
        "model": "openrouter/auto",
        "messages": [
            {"role": "system", "content": mental_health_prompt},
            {"role": "user", "content": user_input}
        ]
    }

    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers=HEADERS,
        json=payload
    )

    result = response.json()
    if 'choices' in result:
        return result['choices'][0]['message']['content']
    return "I'm here for you. Please try again."

print("Enhanced Chatbot Ready!")

Enhanced Chatbot Ready!


In [ ]:
print("Mental Health Support Chatbot")
print("Type 'quit' to exit")
print("=" * 40)

while True:
    user_input = input("You: ")
    if user_input.lower() == 'quit':
        print("Take care! You are stronger than you think.")
        break
    response = mental_health_chatbot_v2(user_input)
    print(f"\nBot: {response}")
    print("-" * 40)

Mental Health Support Chatbot
Type 'quit' to exit
You: i m feeling so sad

Bot:  I'm really sorry that you're feeling sad right now. It's completely normal to have ups and downs in life, but I understand that it can be difficult to go through this feeling. Here are some practical things you can try that might help:

1. Reach out to a trusted friend or family member and share how you're feeling. Sometimes just talking about what's on your mind can help lighten the load.
2. Engage in activities that bring you joy or comfort. This could be reading a book, taking a walk, listening to music, or anything else that makes you feel good.
3. Practice self-care. This could include things like getting enough sleep, eating well, and exercising regularly.
4. Try writing in a journal. Putting your thoughts and feelings down on paper can be a helpful way to process them.

If you've been feeling sad for a long time or if your feelings are interfering with your ability to function in your daily life, it